# NB-Step3 · Remap COCO Annotations to Preprocessed Frame Space
**Pipeline position:** Step 3 of 9 — runs after NB-Step2, produces aligned ground truth for retraining.

### The problem being solved
The original `in_df_147_01.json` contains 859 annotations with coordinates in
**original 2160×3840 video space**.  The preprocessed frames produced by NB-Step2
are **1080×1475 px** (ROI crop → grayscale → 0.5× bicubic → CLAHE).  Training the
U-Net or YOLO on mismatched coordinate spaces produces the temporal-mismatch failure
documented in the paper.  This notebook eliminates that mismatch permanently.

### Transform applied to every coordinate
```
x_p = (x_orig - roi_x0) * scale_x       # roi_x0 = 0,   scale_x = 0.5
y_p = (y_orig - roi_y0) * scale_y       # roi_y0 = 450, scale_y = 0.5
w_p = w_orig * scale_x
h_p = h_orig * scale_y
```
Applied to: `bbox [x, y, w, h]` and every `(x, y)` pair in `segmentation` polygons.

### Outputs
| File | Description |
|------|-------------|
| `in_df_147_01_preproc.json` | Remapped COCO JSON — direct input to U-Net / YOLO training |
| `remap_report_<ts>.json` | Per-image annotation counts, discard reasons |
| `qa_remap_<ts>.png` | Visual overlay: remapped polygons on preprocessed PNGs |


In [ ]:
# ── Cell 1 · Imports ─────────────────────────────────────────────────────────
import cv2
import numpy as np
import json, os, copy
from pathlib import Path
from datetime import datetime
from collections import defaultdict

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Polygon as MplPolygon
from matplotlib.collections import PatchCollection

print(f"✓ Imports complete  |  OpenCV {cv2.__version__}  |  NumPy {np.__version__}")


In [ ]:
# ── Cell 2 · Mount Google Drive ──────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
print("✓ Drive mounted")


In [ ]:
# ── Cell 3 · Static Configuration ───────────────────────────────────────────
# HUMAN-EDITED section — paths only.

ANNOT_BASE = "/content/drive/MyDrive/SIGNAL_NN_2026/HIDRO_2026/anot_pool"

COCO_IN          = f"{ANNOT_BASE}/in_df_147_01.json"
PREPROC_MANIFEST = None   # auto-resolved below from most recent file in preproc_frames_147
PREPROC_FRAMES   = f"{ANNOT_BASE}/preproc_frames_147"
COCO_OUT         = f"{ANNOT_BASE}/in_df_147_01_preproc.json"

STEP3_TS = datetime.now().strftime("%Y%m%d_%H%M%S")

# Auto-resolve most recent preproc_manifest if not set above
if PREPROC_MANIFEST is None:
    candidates = sorted([
        f for f in os.listdir(PREPROC_FRAMES)
        if f.startswith("preproc_manifest_") and f.endswith(".json")
    ])
    if not candidates:
        raise FileNotFoundError(
            f"No preproc_manifest_*.json found in {PREPROC_FRAMES}\n"
            "Run NB-Step2 first, or set PREPROC_MANIFEST manually above."
        )
    PREPROC_MANIFEST = os.path.join(PREPROC_FRAMES, candidates[-1])
    print(f"  Auto-resolved preproc_manifest: {candidates[-1]}")

print("✓ Cell 3 — configuration loaded")
print(f"  COCO input  : {COCO_IN}")
print(f"  COCO output : {COCO_OUT}")
print(f"  Preproc dir : {PREPROC_FRAMES}")


In [ ]:
# ── Cell 4 · Load Inputs ─────────────────────────────────────────────────────

# ── COCO JSON ─────────────────────────────────────────────────────────────────
with open(COCO_IN) as f:
    coco_orig = json.load(f)

n_images = len(coco_orig["images"])
n_annots = len(coco_orig["annotations"])
print(f"✓ COCO loaded  |  {n_images} images  |  {n_annots} annotations")

# ── Preproc manifest ──────────────────────────────────────────────────────────
with open(PREPROC_MANIFEST) as f:
    pm = json.load(f)

PREPROC_W  = pm["preproc_w"]
PREPROC_H  = pm["preproc_h"]
ROI        = pm["roi"]               # [x0, y0, x1, y1]
SCALE      = pm["downscale"]         # 0.5

ROI_X0, ROI_Y0 = ROI[0], ROI[1]

print(f"✓ Preproc manifest loaded")
print(f"  ROI          : {ROI}")
print(f"  Preprocessed : {PREPROC_W}×{PREPROC_H} px")
print(f"  Scale        : {SCALE}")
print(f"  Transform    : x_p = (x - {ROI_X0}) × {SCALE}")
print(f"                 y_p = (y - {ROI_Y0}) × {SCALE}")

# ── Build stem → output_png lookup from preproc manifest ─────────────────────
stem_to_png = {
    rec["stem"]: rec["output_png"]
    for rec in pm["frames"]
    if rec["verified"] == "ok"
}
print(f"  Verified PNGs: {len(stem_to_png)}")

# ── Build image_id → image record lookup ─────────────────────────────────────
id_to_image = {img["id"]: img for img in coco_orig["images"]}


In [ ]:
# ── Cell 5 · Remap Functions ─────────────────────────────────────────────────

def remap_coord(x: float, y: float) -> tuple:
    """Remap a single (x, y) point from original video space to preprocessed space."""
    return (x - ROI_X0) * SCALE, (y - ROI_Y0) * SCALE


def remap_bbox(bbox: list) -> list:
    """Remap COCO bbox [x, y, w, h] → preprocessed space."""
    x, y, w, h = bbox
    xp, yp = remap_coord(x, y)
    return [round(xp, 4), round(yp, 4),
            round(w * SCALE, 4), round(h * SCALE, 4)]


def remap_segmentation(seg: list) -> list:
    """
    Remap a COCO segmentation polygon.
    Input:  [[x1,y1,x2,y2,...]]  (list of flat coordinate lists)
    Output: same structure with remapped coordinates.
    """
    remapped = []
    for poly in seg:
        new_poly = []
        for i in range(0, len(poly), 2):
            xp, yp = remap_coord(poly[i], poly[i+1])
            new_poly.extend([round(xp, 4), round(yp, 4)])
        remapped.append(new_poly)
    return remapped


def is_inside_preproc(bbox_p: list) -> bool:
    """
    Return True if the bbox centre lies within the preprocessed frame.
    Annotations whose centre is outside are discarded.
    """
    x, y, w, h = bbox_p
    cx = x + w / 2
    cy = y + h / 2
    return 0 <= cx <= PREPROC_W and 0 <= cy <= PREPROC_H


def bbox_to_area(bbox: list) -> float:
    return round(bbox[2] * bbox[3], 4)


print("✓ Cell 5 — remap functions defined")
print(f"  is_inside_preproc valid range: [0,{PREPROC_W}] × [0,{PREPROC_H}]")


In [ ]:
# ── Cell 6 · Apply Remap & Filter ────────────────────────────────────────────

new_images      = []
new_annotations = []
remap_report    = []

n_kept     = 0
n_discarded = 0

# Index annotations by image_id for fast lookup
annots_by_image = defaultdict(list)
for ann in coco_orig["annotations"]:
    annots_by_image[ann["image_id"]].append(ann)

for img in coco_orig["images"]:
    stem     = Path(img["file_name"]).stem   # VID_20260429_124526_frame00120
    png_path = stem_to_png.get(stem)

    if png_path is None:
        # Image was not successfully preprocessed in NB-Step2 — skip entirely
        remap_report.append({
            "image_id":   img["id"],
            "stem":       stem,
            "status":     "png_missing",
            "n_kept":     0,
            "n_discarded": len(annots_by_image[img["id"]]),
            "discards":   [],
        })
        n_discarded += len(annots_by_image[img["id"]])
        continue

    # Update image record
    new_img = copy.deepcopy(img)
    new_img["file_name"] = f"{stem}.png"
    new_img["width"]     = PREPROC_W
    new_img["height"]    = PREPROC_H
    new_images.append(new_img)

    kept_count      = 0
    discarded_count = 0
    discard_reasons = []

    for ann in annots_by_image[img["id"]]:
        bbox_p = remap_bbox(ann["bbox"])

        if not is_inside_preproc(bbox_p):
            discarded_count += 1
            n_discarded     += 1
            discard_reasons.append({
                "ann_id": ann["id"],
                "reason": "centre_outside_preproc",
                "orig_bbox": ann["bbox"],
                "remap_bbox": bbox_p,
            })
            continue

        new_ann                = copy.deepcopy(ann)
        new_ann["bbox"]        = bbox_p
        new_ann["segmentation"]= remap_segmentation(ann["segmentation"])
        new_ann["area"]        = bbox_to_area(bbox_p)
        new_annotations.append(new_ann)
        kept_count += 1
        n_kept     += 1

    remap_report.append({
        "image_id":    img["id"],
        "stem":        stem,
        "status":      "ok",
        "n_kept":      kept_count,
        "n_discarded": discarded_count,
        "discards":    discard_reasons,
    })

# ── Summary ───────────────────────────────────────────────────────────────────
W = 66
print("=" * W)
print("  Cell 6 — Remap Results".center(W))
print("=" * W)
print(f"  Images processed  : {len(new_images)} / {n_images}")
print(f"  Annotations kept  : {n_kept} / {n_annots}"
      f"  ({100*n_kept/n_annots:.1f} %)")
print(f"  Annotations discarded: {n_discarded}"
      f"  ({100*n_discarded/n_annots:.1f} %)")

if n_discarded > 0:
    print()
    print("  Discard breakdown:")
    for rep in remap_report:
        if rep["n_discarded"] > 0:
            print(f"    {rep['stem']:<45}  "
                  f"{rep['n_discarded']:>3} discarded  "
                  f"{rep['n_kept']:>3} kept")
print("=" * W)


In [ ]:
# ── Cell 7 · Visual QA — Overlay Remapped Annotations on Preprocessed PNGs ──
# Draws remapped segmentation polygons over a sample of preprocessed frames.
# Green = annotation kept.  If polygons align with bubble positions: remap is correct.

import random

ok_reports = [r for r in remap_report if r["status"] == "ok" and r["n_kept"] > 0]
sample     = sorted(random.sample(ok_reports, min(12, len(ok_reports))),
                    key=lambda x: x["stem"])

# Build ann lookup by image_id for the remapped set
new_ann_by_imgid = defaultdict(list)
for ann in new_annotations:
    new_ann_by_imgid[ann["image_id"]].append(ann)

n_cols = 4
n_rows = (len(sample) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=(n_cols * 3.0, n_rows * 4.2))
axes = np.array(axes).flatten()

for i, rep in enumerate(sample):
    ax      = axes[i]
    img_id  = rep["image_id"]
    png_path = stem_to_png[rep["stem"]]
    frame   = cv2.imread(png_path, cv2.IMREAD_GRAYSCALE)
    ax.imshow(frame, cmap="gray", aspect="auto")

    patches_list = []
    for ann in new_ann_by_imgid[img_id]:
        for poly in ann["segmentation"]:
            pts = np.array(poly).reshape(-1, 2)
            patches_list.append(MplPolygon(pts, closed=True))

    if patches_list:
        pc = PatchCollection(patches_list, facecolor="none",
                             edgecolor="lime", linewidths=0.8)
        ax.add_collection(pc)

    n_ann = rep["n_kept"]
    ax.set_title(f"{rep['stem'][-16:]}\n{n_ann} ann",
                 fontsize=7, color="lime" if n_ann > 0 else "tomato", pad=2)
    ax.axis("off")

for j in range(len(sample), len(axes)):
    axes[j].axis("off")

fig.suptitle(
    "NB-Step3 QA — remapped segmentation polygons (green) on preprocessed frames\n"
    "Polygons must align with visible bubbles — if they do, the remap is correct.",
    fontsize=9, fontweight="bold"
)
plt.tight_layout(pad=0.4)
qa_path = os.path.join(PREPROC_FRAMES, f"qa_remap_{STEP3_TS}.png")
plt.savefig(qa_path, dpi=120, bbox_inches="tight")
plt.show()
plt.close()
print(f"✓ QA panel saved: {qa_path}")
print()
print("  Inspect the panel carefully.")
print("  Green polygons must sit ON the bright bubble blobs.")
print("  If they are offset, check ROI_X0 / ROI_Y0 in Cell 4.")


In [ ]:
# ── Cell 8 · Export Remapped COCO JSON ───────────────────────────────────────

coco_out = {
    "licenses":    coco_orig.get("licenses",  []),
    "info":        coco_orig.get("info",      {}),
    "categories":  coco_orig["categories"],
    "images":      new_images,
    "annotations": new_annotations,
}

with open(COCO_OUT, "w") as f:
    json.dump(coco_out, f, indent=2)
print(f"✓ Remapped COCO JSON saved: {COCO_OUT}")

# ── Remap report ──────────────────────────────────────────────────────────────
report_path = os.path.join(PREPROC_FRAMES, f"remap_report_{STEP3_TS}.json")
report_doc  = {
    "generated_at":    STEP3_TS,
    "coco_in":         COCO_IN,
    "coco_out":        COCO_OUT,
    "roi":             ROI,
    "scale":           SCALE,
    "preproc_w":       PREPROC_W,
    "preproc_h":       PREPROC_H,
    "n_images_in":     n_images,
    "n_images_out":    len(new_images),
    "n_annotations_in":  n_annots,
    "n_annotations_out": n_kept,
    "n_discarded":     n_discarded,
    "per_image":       remap_report,
}
with open(report_path, "w") as f:
    json.dump(report_doc, f, indent=2)
print(f"✓ Remap report saved: {report_path}")

# ── Final summary ─────────────────────────────────────────────────────────────
W = 66
print()
print("=" * W)
print("  NB-Step3 · SUMMARY".center(W))
print("=" * W)
print(f"  Input  COCO  : {n_images} images, {n_annots} annotations")
print(f"  Output COCO  : {len(new_images)} images, {n_kept} annotations")
print(f"  Discarded    : {n_discarded} annotations  "
      f"({100*n_discarded/n_annots:.1f} %)")
print(f"  Mean ann/img : {n_kept/len(new_images):.1f}  (was {n_annots/n_images:.1f})")
print()
print(f"  Coordinate space : 2160×3840 px  →  {PREPROC_W}×{PREPROC_H} px")
print(f"  Transform        : x_p = (x - {ROI_X0}) × {SCALE}")
print(f"                     y_p = (y - {ROI_Y0}) × {SCALE}")
print()
print(f"  Output file  : {COCO_OUT}")
print()
print("  ✓ Step 3 complete.")
print("  Next → NB-Step4: retrain U-Net on aligned preprocessed patches.")
print("  Key input: in_df_147_01_preproc.json  +  preproc_frames_147/")
print("=" * W)


In [ ]:
import json

COCO_PREPROC = "/content/drive/MyDrive/SIGNAL_NN_2026/HIDRO_2026/anot_pool/in_df_147_01_preproc.json"

with open(COCO_PREPROC) as f:
    coco = json.load(f)

n_images = len(coco["images"])
n_annots = len(coco["annotations"])
ann_per_img = n_annots / n_images

# Bbox size distribution in preprocessed space
import numpy as np
widths  = [a["bbox"][2] for a in coco["annotations"]]
heights = [a["bbox"][3] for a in coco["annotations"]]

print(f"Images      : {n_images}")
print(f"Annotations : {n_annots}")
print(f"Ann/image   : {ann_per_img:.1f}")
print(f"\nBbox width  (px)  min={min(widths):.1f}  max={max(widths):.1f}  "
      f"mean={np.mean(widths):.1f}  median={np.median(widths):.1f}")
print(f"Bbox height (px)  min={min(heights):.1f}  max={max(heights):.1f}  "
      f"mean={np.mean(heights):.1f}  median={np.median(heights):.1f}")